### Notebook 2 - Desarrollo e implementación del pipeline técnico

Simulación Monte Carlo multivariante del valor del colateral, motor de financiación (línea de crédito / préstamo amortizable), entrenamiento y evaluación de modelos de Machine Learning (clasificación de margin call y regresión del LTV máximo P95), optimización de carteras y cálculo de todas las tablas agregadas usadas en la memoria y en el Notebook 3 de gráficos.

**Entrada:** `data/dataset_mercado.parquet`, `data/catalogo_carteras.parquet`, `data/dataset_colateral.parquet` (generados por el Notebook 1).

**Salida:** todos los datasets intermedios, resultados de modelos, tablas agregadas (`tabla_*.parquet`) y modelos entrenados (`.joblib`), guardados en `data/`.

## Carga de entradas (salida del Notebook 1)

In [1]:
import pandas as pd
import numpy as np

dataset_mercado = pd.read_parquet("data/entradas/dataset_mercado.parquet")
catalogo_carteras = pd.read_parquet("data/entradas/catalogo_carteras.parquet")
dataset_colateral = pd.read_parquet("data/entradas/dataset_colateral.parquet")

print("dataset_mercado:", dataset_mercado.shape)
print("catalogo_carteras:", catalogo_carteras.shape)
print("dataset_colateral:", dataset_colateral.shape)


dataset_mercado: (2097, 45)
catalogo_carteras: (286, 8)
dataset_colateral: (599742, 25)


In [2]:
# ============================================================
# Constantes compartidas (idénticas a las del Notebook 1,
# necesarias aquí para los motores de simulación y financiación)
# ============================================================

NOMBRE_ACTIVOS = ["XEON.DE", "SHY", "AGGU.L", "SPY"]

TER = {
    "XEON.DE": 0.0010,
    "SHY": 0.0015,
    "AGGU.L": 0.0010,
    "SPY": 0.0009
}

SPREAD = 0.025
EURIBOR_3M_ASSUMED = 0.035


In [3]:
# ============================================================
# 9. Motor Monte Carlo multivariado
# ============================================================

N_SIMULATIONS = 1000
HORIZON_DAYS = 252 * 3
RANDOM_SEED = 42
INITIAL_COLLATERAL = 100_000


def motor_montecarlo_multivariado(
    dataset_mercado,
    catalogo_carteras,
    dataset_colateral,
    tickers,
    n_simulations=1000,
    horizon_days=756,
    seed=42,
    initial_collateral=100_000,
    chunk_size=20,
):
    """Simulación Monte Carlo multivariada de trayectorias de valor del colateral.

    El cálculo se realiza en lotes ("chunks") de carteras en lugar de construir
    de una sola vez el tensor completo simulaciones x días x carteras: para
    n_simulations=1000, horizon_days=756 y 286 carteras ese tensor ocuparía
    más de 10 GB en memoria. Procesar por lotes de `chunk_size` carteras
    produce exactamente el mismo resultado (la operación es independiente por
    cartera) con un pico de memoria varios órdenes de magnitud menor.
    """
    rng = np.random.default_rng(seed)

    ret_cols = [f"ret_1d_{t}" for t in tickers]
    weight_cols = [f"w_{t}" for t in tickers]

    retornos_hist = dataset_mercado[ret_cols].dropna().copy()
    mu = retornos_hist.mean().to_numpy()
    cov = retornos_hist.cov().to_numpy()
    drift = mu - 0.5 * np.diag(cov)

    shocks = rng.multivariate_normal(
        mean=drift,
        cov=cov,
        size=(n_simulations, horizon_days)
    )

    carteras = catalogo_carteras.copy().reset_index(drop=True)
    W = carteras[weight_cols].to_numpy()

    resumen_hist = (
        dataset_colateral
        .dropna()
        .groupby("id_cartera")
        .agg(
            vol_252d_cartera=("vol_252d_cartera", "last"),
            drawdown_63d_cartera=("drawdown_63d_cartera", "last"),
            var_63d_95=("var_63d_95", "last"),
            es_63d_95=("es_63d_95", "last"),
            rentabilidad_neta_252d=("rentabilidad_neta_252d", "last")
        )
        .reset_index()
    )

    registros = []
    n_c = len(carteras)
    for start in range(0, n_c, chunk_size):
        end = min(start + chunk_size, n_c)
        W_chunk = W[start:end]

        retornos_carteras = np.einsum("sda,ca->sdc", shocks, W_chunk)
        trayectorias = initial_collateral * np.exp(np.cumsum(retornos_carteras, axis=1))

        valor_final = trayectorias[:, -1, :]
        valor_minimo = trayectorias.min(axis=1)
        valor_maximo = trayectorias.max(axis=1)
        dia_valor_minimo = trayectorias.argmin(axis=1) + 1

        retorno_simulado = valor_final / initial_collateral - 1
        drawdown_simulado = valor_minimo / valor_maximo - 1

        for local_idx in range(end - start):
            idx = start + local_idx
            cartera = carteras.iloc[idx]
            temp = pd.DataFrame({
                "id_cartera": cartera["id_cartera"],
                "id_simulacion": np.arange(1, n_simulations + 1),
                "perfil": cartera["perfil"],

                "valor_colateral_inicial": initial_collateral,
                "valor_colateral_final": valor_final[:, local_idx],
                "valor_colateral_minimo": valor_minimo[:, local_idx],
                "valor_colateral_maximo": valor_maximo[:, local_idx],
                "dia_valor_minimo": dia_valor_minimo[:, local_idx],

                "retorno_simulado": retorno_simulado[:, local_idx],
                "drawdown_simulado": drawdown_simulado[:, local_idx],

                "hhi_concentracion": cartera["hhi_concentracion"],
                "ter_cartera": cartera["ter_cartera"],

                "w_XEON.DE": cartera["w_XEON.DE"],
                "w_SHY": cartera["w_SHY"],
                "w_AGGU.L": cartera["w_AGGU.L"],
                "w_SPY": cartera["w_SPY"],
            })
            registros.append(temp)

        del retornos_carteras, trayectorias, valor_final, valor_minimo, valor_maximo, dia_valor_minimo, retorno_simulado, drawdown_simulado

    dataset_montecarlo = pd.concat(registros, ignore_index=True)
    dataset_montecarlo = dataset_montecarlo.merge(resumen_hist, on="id_cartera", how="left")
    return dataset_montecarlo


In [4]:
dataset_montecarlo = motor_montecarlo_multivariado(
    dataset_mercado=dataset_mercado,
    catalogo_carteras=catalogo_carteras,
    dataset_colateral=dataset_colateral,
    tickers=NOMBRE_ACTIVOS,
    n_simulations=N_SIMULATIONS,
    horizon_days=HORIZON_DAYS,
    seed=RANDOM_SEED,
    initial_collateral=INITIAL_COLLATERAL,
    chunk_size=20
)

print(dataset_montecarlo.shape)
display(dataset_montecarlo.head())
display(dataset_montecarlo.describe())
display(dataset_montecarlo["valor_colateral_inicial"].describe())

(286000, 21)


,id_cartera,id_simulacion,perfil,valor_colateral_inicial,valor_colateral_final,valor_colateral_minimo,valor_colateral_maximo,dia_valor_minimo,retorno_simulado,drawdown_simulado,...,ter_cartera,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,vol_252d_cartera,drawdown_63d_cartera,var_63d_95,es_63d_95,rentabilidad_neta_252d
0,1,1,dinamico,100000,213896.077821,99529.225077,213896.077821,1,1.138961,-0.534684,...,0.0009,0.0,0.0,0.0,1.0,0.135509,-0.000472,-0.013779,-0.017766,0.204638
1,1,2,dinamico,100000,106818.201869,84788.875878,119048.022768,192,0.068182,-0.287776,...,0.0009,0.0,0.0,0.0,1.0,0.135509,-0.000472,-0.013779,-0.017766,0.204638
2,1,3,dinamico,100000,156311.869366,74925.103060,159622.347638,122,0.563119,-0.530610,...,0.0009,0.0,0.0,0.0,1.0,0.135509,-0.000472,-0.013779,-0.017766,0.204638
3,1,4,dinamico,100000,285409.165349,99575.874530,289269.389572,1,1.854092,-0.655768,...,0.0009,0.0,0.0,0.0,1.0,0.135509,-0.000472,-0.013779,-0.017766,0.204638
4,1,5,dinamico,100000,228430.126500,99230.987094,241654.694952,5,1.284301,-0.589369,...,0.0009,0.0,0.0,0.0,1.0,0.135509,-0.000472,-0.013779,-0.017766,0.204638


,id_cartera,id_simulacion,valor_colateral_inicial,valor_colateral_final,valor_colateral_minimo,valor_colateral_maximo,dia_valor_minimo,retorno_simulado,drawdown_simulado,hhi_concentracion,ter_cartera,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,vol_252d_cartera,drawdown_63d_cartera,var_63d_95,es_63d_95,rentabilidad_neta_252d
count,286000.000000,286000.000000,286000.0,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.00000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000,286000.000000
mean,143.500000,500.500000,100000.0,117872.154248,95040.263777,124038.614199,185.596888,0.178722,-0.214719,0.46000,0.001100,0.250000,0.250000,0.250000,0.250000,0.060224,-0.000471,-0.005221,-0.006637,0.035172
std,82.560728,288.675495,0.0,23476.739831,5213.168755,23614.115081,217.794291,0.234767,0.111691,0.14697,0.000124,0.229129,0.229129,0.229129,0.229129,0.023024,0.000776,0.002284,0.002973,0.050860
min,1.000000,1.000000,100000.0,51710.107936,41485.817398,98930.907387,1.000000,-0.482899,-0.848142,0.26000,0.000900,0.000000,0.000000,0.000000,0.000000,0.002195,-0.002781,-0.013779,-0.017766,-0.040453
25%,72.000000,250.750000,100000.0,103667.582682,92920.992544,108731.240078,18.000000,0.036676,-0.275405,0.36000,0.001000,0.100000,0.100000,0.100000,0.100000,0.045459,-0.000733,-0.006318,-0.007908,-0.007087
50%,143.500000,500.500000,100000.0,112124.032490,96622.294416,117269.096321,86.000000,0.121240,-0.191222,0.42000,0.001080,0.200000,0.200000,0.200000,0.200000,0.059651,0.000000,-0.005026,-0.006246,0.024422
75%,215.000000,750.250000,100000.0,125956.439374,98785.354700,131495.825436,280.000000,0.259564,-0.133767,0.54000,0.001180,0.400000,0.400000,0.400000,0.400000,0.072392,0.000000,-0.003790,-0.004786,0.067428
max,286.000000,1000.000000,100000.0,624290.553356,103572.734382,649161.995817,756.000000,5.242906,-0.010464,1.00000,0.001500,1.000000,1.000000,1.000000,1.000000,0.135509,0.000000,-0.000067,-0.000117,0.204638


count    286000.0
mean     100000.0
std           0.0
min      100000.0
25%      100000.0
50%      100000.0
75%      100000.0
max      100000.0
Name: valor_colateral_inicial, dtype: float64

In [5]:
dataset_montecarlo["valor_colateral_inicial"].describe()

count    286000.0
mean     100000.0
std           0.0
min      100000.0
25%      100000.0
50%      100000.0
75%      100000.0
max      100000.0
Name: valor_colateral_inicial, dtype: float64

In [6]:
# ============================================================
# 10. Motor Financiero mejorado
# ============================================================

LTV_INICIAL = 0.60
MARGIN_CALL_THRESHOLD = 0.80
EURIBOR_3M_ASSUMED = 0.035
SPREAD = 0.025
LOAN_YEARS = 3
TRADING_DAYS = 252
HORIZON_DAYS = LOAN_YEARS * TRADING_DAYS


def motor_financiacion(dataset_montecarlo):
    registros = []

    tasa_financiacion = EURIBOR_3M_ASSUMED + SPREAD
    deuda_inicial_base = INITIAL_COLLATERAL * LTV_INICIAL

    for tipo in ["linea_credito", "prestamo_amortizable"]:
        df = dataset_montecarlo.copy()

        df["tipo_financiacion"] = tipo
        df["ltv_inicial"] = LTV_INICIAL
        df["euribor_3m"] = EURIBOR_3M_ASSUMED
        df["spread"] = SPREAD
        df["coste_financiacion"] = tasa_financiacion

        df["deuda_inicial"] = deuda_inicial_base

        # Día en que ocurre el peor valor de colateral
        df["dia_valor_minimo"] = df["dia_valor_minimo"].clip(lower=1, upper=HORIZON_DAYS)

        if tipo == "linea_credito":
            # Principal constante
            df["saldo_deuda_final"] = df["deuda_inicial"]
            df["saldo_deuda_en_minimo"] = df["deuda_inicial"]
            df["saldo_deuda_promedio"] = df["deuda_inicial"]

        elif tipo == "prestamo_amortizable":
            # Amortización lineal aproximada
            df["saldo_deuda_final"] = 0.0

            df["saldo_deuda_en_minimo"] = df["deuda_inicial"] * (
                1 - df["dia_valor_minimo"] / HORIZON_DAYS
            )

            df["saldo_deuda_en_minimo"] = df["saldo_deuda_en_minimo"].clip(lower=0)

            df["saldo_deuda_promedio"] = df["deuda_inicial"] / 2

        # Coste financiero estimado durante el horizonte
        df["interes_estimado_total"] = (
            df["saldo_deuda_promedio"] * tasa_financiacion * LOAN_YEARS
        )

        df["interes_estimado_pct_colateral"] = (
            df["interes_estimado_total"] / df["valor_colateral_inicial"]
        )

        # LTVs
        df["ltv_final"] = df["saldo_deuda_final"] / df["valor_colateral_final"]

        df["ltv_maximo_aproximado"] = (
            df["saldo_deuda_en_minimo"] / df["valor_colateral_minimo"]
        )

        df["exceso_ltv_sobre_umbral"] = (
            df["ltv_maximo_aproximado"] - MARGIN_CALL_THRESHOLD
        )

        df["exceso_ltv_sobre_umbral"] = df["exceso_ltv_sobre_umbral"].clip(lower=0)

        # Target
        df["margin_call"] = np.where(
            df["ltv_maximo_aproximado"] >= MARGIN_CALL_THRESHOLD,
            1,
            0
        )

        # Severidad del margin call
        df["severidad_margin_call"] = np.where(
            df["margin_call"] == 1,
            df["exceso_ltv_sobre_umbral"],
            0
        )

        # Distancia de seguridad al umbral
        df["buffer_ltv"] = (
            MARGIN_CALL_THRESHOLD - df["ltv_maximo_aproximado"]
        )

        # Rentabilidad neta del cliente
        df["rentabilidad_neta_simulada"] = (
            df["retorno_simulado"]
            - df["interes_estimado_pct_colateral"]
            - df["ter_cartera"]
        )

        # Indicador de pérdida económica neta
        df["resultado_neto_negativo"] = np.where(
            df["rentabilidad_neta_simulada"] < 0,
            1,
            0
        )

        registros.append(df)

    dataset_financiacion = pd.concat(registros, ignore_index=True)

    return dataset_financiacion

In [7]:
dataset_financiacion = motor_financiacion(dataset_montecarlo)

print(dataset_financiacion.shape)

display(dataset_financiacion["margin_call"].value_counts(normalize=True))

display(
    dataset_financiacion
    .groupby("tipo_financiacion")
    .agg(
        prob_margin_call=("margin_call", "mean"),
        ltv_maximo_medio=("ltv_maximo_aproximado", "mean"),
        severidad_media=("severidad_margin_call", "mean"),
        buffer_ltv_medio=("buffer_ltv", "mean"),
        rentabilidad_neta_media=("rentabilidad_neta_simulada", "mean"),
        prob_resultado_neto_negativo=("resultado_neto_negativo", "mean")
    )
)

(572000, 40)


margin_call
0    0.99622
1    0.00378
Name: proportion, dtype: float64

,prob_margin_call,ltv_maximo_medio,severidad_media,buffer_ltv_medio,rentabilidad_neta_media,prob_resultado_neto_negativo
tipo_financiacion,,,,,,
linea_credito,0.007559,0.633477,0.00048,0.166523,0.069622,0.469028
prestamo_amortizable,0.000000,0.471965,0.00000,0.328035,0.123622,0.311538


In [8]:
# ============================================================
# 11. Motor Machine Learning
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    classification_report
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False


# ============================================================
# 11.1 Preparación dataset ML sin data leakage
# ============================================================

def preparar_dataset_ml(dataset_financiacion):
    """
    Dataset ML sin variables futuras.
    Excluye:
    - retorno_simulado
    - drawdown_simulado
    - valor_colateral_final
    - valor_colateral_minimo
    - valor_colateral_maximo
    - ltv_maximo_aproximado
    """

    features = [
        # Riesgo histórico conocido antes de simular
        "vol_252d_cartera",
        "drawdown_63d_cartera",
        "var_63d_95",
        "es_63d_95",

        # Composición de cartera
        "hhi_concentracion",
        "ter_cartera",
        "w_XEON.DE",
        "w_SHY",
        "w_AGGU.L",
        "w_SPY",

        # Rentabilidad histórica neta
        "rentabilidad_neta_252d",

        # Financiación conocida al inicio
        "ltv_inicial",
        "coste_financiacion",
        "deuda_inicial",

        # Variables categóricas
        "tipo_financiacion",
        "perfil"
    ]

    target = "margin_call"

    df = dataset_financiacion[features + [target]].dropna().copy()

    X = df[features]
    y = df[target]

    return X, y, features


def evaluar_modelo(nombre, modelo, X_test, y_test):
    y_proba = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    return {
        "modelo": nombre,
        "auc_roc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_test, y_proba)
    }


def motor_machine_learning(dataset_financiacion):
    X, y, features = preparar_dataset_ml(dataset_financiacion)

    numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )

    modelos = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    }

    if XGBOOST_AVAILABLE:
        modelos["XGBoost"] = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

    resultados = []
    modelos_entrenados = {}

    for nombre, modelo_base in modelos.items():
        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", modelo_base)
            ]
        )

        pipeline.fit(X_train, y_train)

        resultados.append(
            evaluar_modelo(nombre, pipeline, X_test, y_test)
        )

        modelos_entrenados[nombre] = pipeline

    df_resultados = pd.DataFrame(resultados).sort_values("pr_auc", ascending=False)

    return df_resultados, modelos_entrenados, X_train, X_test, y_train, y_test

In [9]:
df_resultados_ml, modelos_entrenados, X_train, X_test, y_train, y_test = motor_machine_learning(
    dataset_financiacion
)

display(df_resultados_ml)

,modelo,auc_roc,pr_auc,precision,recall,f1,brier_score
0,Logistic Regression,0.935336,0.050412,0.019114,0.898305,0.037432,0.118818
1,Random Forest,0.931415,0.048423,0.017660,0.885978,0.034630,0.114228
2,XGBoost,0.930935,0.048283,0.000000,0.000000,0.000000,0.003662


2. motor ML de regresión

In [10]:
# dataset_financiacion = pd.read_parquet("data/salidas/dataset_financiacion.parquet")
# catalogo_carteras = pd.read_parquet("data/entradas/catalogo_carteras.parquet")

In [11]:
# ============================================================
# 11B. ML de regresión del riesgo LTV
# ============================================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

try:
    from xgboost import XGBRegressor
    XGBOOST_REG_AVAILABLE = True
except ImportError:
    XGBOOST_REG_AVAILABLE = False

Preparación del dataset de regresión

In [12]:
def construir_dataset_regresion_ltv(dataset_financiacion):
    """
    Construye un dataset agregado por cartera y tipo de financiación.

    Targets generados:
    - ltv_maximo_medio
    - ltv_maximo_p95
    - prob_margin_call

    Se recomienda utilizar ltv_maximo_p95 como objetivo principal,
    al representar un escenario adverso de riesgo.
    """

    columnas_grupo = [
        "id_cartera",
        "perfil",
        "tipo_financiacion",

        "w_XEON.DE",
        "w_SHY",
        "w_AGGU.L",
        "w_SPY",

        "hhi_concentracion",
        "ter_cartera",

        "vol_252d_cartera",
        "drawdown_63d_cartera",
        "var_63d_95",
        "es_63d_95",
        "rentabilidad_neta_252d",

        "ltv_inicial",
        "coste_financiacion",
        "deuda_inicial"
    ]

    dataset_regresion = (
        dataset_financiacion
        .groupby(columnas_grupo, dropna=False)
        .agg(
            ltv_maximo_medio=(
                "ltv_maximo_aproximado",
                "mean"
            ),
            ltv_maximo_p50=(
                "ltv_maximo_aproximado",
                "median"
            ),
            ltv_maximo_p95=(
                "ltv_maximo_aproximado",
                lambda x: x.quantile(0.95)
            ),
            ltv_maximo_p99=(
                "ltv_maximo_aproximado",
                lambda x: x.quantile(0.99)
            ),
            prob_margin_call=(
                "margin_call",
                "mean"
            ),
            buffer_ltv_medio=(
                "buffer_ltv",
                "mean"
            ),
            severidad_media=(
                "severidad_margin_call",
                "mean"
            ),
            rentabilidad_neta_media=(
                "rentabilidad_neta_simulada",
                "mean"
            ),
            drawdown_simulado_medio=(
                "drawdown_simulado",
                "mean"
            )
        )
        .reset_index()
    )

    return dataset_regresion

3. Preparar variables y target

In [13]:
def preparar_dataset_ml_regresion(
    dataset_regresion,
    target="ltv_maximo_p95"
):
    """
    Prepara el dataset de regresión sin variables futuras directas.

    Target recomendado:
        ltv_maximo_p95

    También admite:
        ltv_maximo_medio
        prob_margin_call
    """

    features = [
        # Riesgo histórico conocido al originar la operación
        "vol_252d_cartera",
        "drawdown_63d_cartera",
        "var_63d_95",
        "es_63d_95",
        "rentabilidad_neta_252d",

        # Composición
        "hhi_concentracion",
        "ter_cartera",
        "w_XEON.DE",
        "w_SHY",
        "w_AGGU.L",
        "w_SPY",

        # Condiciones financieras conocidas al inicio
        "ltv_inicial",
        "coste_financiacion",
        "deuda_inicial",

        # Variables categóricas
        "tipo_financiacion",
        "perfil"
    ]

    columnas = [
        "id_cartera",
        *features,
        target
    ]

    df = (
        dataset_regresion[columnas]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    X = df[features]
    y = df[target]
    grupos = df["id_cartera"]

    return X, y, grupos, features

4. Evaluación de regresores

In [14]:
def evaluar_modelo_regresion(
    nombre,
    modelo,
    X_test,
    y_test
):
    y_pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return {
        "modelo": nombre,
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

5. Motor completo de regresión

In [15]:
def motor_machine_learning_regresion(
    dataset_financiacion,
    target="ltv_maximo_p95",
    test_size=0.30,
    random_state=42
):
    # --------------------------------------------
    # 1. Dataset agregado
    # --------------------------------------------
    dataset_regresion = construir_dataset_regresion_ltv(
        dataset_financiacion
    )

    X, y, grupos, features = preparar_dataset_ml_regresion(
        dataset_regresion=dataset_regresion,
        target=target
    )

    # --------------------------------------------
    # 2. Variables numéricas y categóricas
    # --------------------------------------------
    numeric_features = (
        X.select_dtypes(include=np.number)
        .columns
        .tolist()
    )

    categorical_features = (
        X.select_dtypes(exclude=np.number)
        .columns
        .tolist()
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                StandardScaler(),
                numeric_features
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ],
        remainder="drop"
    )

    # --------------------------------------------
    # 3. Split por cartera
    # --------------------------------------------
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=grupos)
    )

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    grupos_train = grupos.iloc[train_idx].copy()
    grupos_test = grupos.iloc[test_idx].copy()

    # Comprobación: no deben existir carteras compartidas
    grupos_comunes = set(grupos_train).intersection(
        set(grupos_test)
    )

    if grupos_comunes:
        raise ValueError(
            "Existe fuga entre train y test: "
            f"{len(grupos_comunes)} carteras compartidas."
        )

    # --------------------------------------------
    # 4. Modelos
    # --------------------------------------------
    modelos = {
        "Random Forest Regressor": RandomForestRegressor(
            n_estimators=400,
            max_depth=12,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=random_state,
            n_jobs=-1
        ),

        "Gradient Boosting Regressor": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=5,
            loss="huber",
            random_state=random_state
        )
    }

    if XGBOOST_REG_AVAILABLE:
        modelos["XGBoost Regressor"] = XGBRegressor(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=random_state,
            n_jobs=-1
        )

    # --------------------------------------------
    # 5. Entrenamiento
    # --------------------------------------------
    resultados = []
    modelos_entrenados = {}

    for nombre, modelo_base in modelos.items():
        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("model", modelo_base)
            ]
        )

        pipeline.fit(X_train, y_train)

        metricas = evaluar_modelo_regresion(
            nombre=nombre,
            modelo=pipeline,
            X_test=X_test,
            y_test=y_test
        )

        resultados.append(metricas)
        modelos_entrenados[nombre] = pipeline

    resultados_regresion = (
        pd.DataFrame(resultados)
        .sort_values(
            ["mae", "rmse"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

    return {
        "dataset_regresion": dataset_regresion,
        "resultados": resultados_regresion,
        "modelos": modelos_entrenados,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "features": features,
        "target": target,
        "grupos_train": grupos_train,
        "grupos_test": grupos_test
    }

6. Ejecutar el motor

In [16]:
resultado_ml_regresion = motor_machine_learning_regresion(
    dataset_financiacion=dataset_financiacion,
    target="ltv_maximo_p95",
    test_size=0.30,
    random_state=42
)

dataset_regresion_ltv = resultado_ml_regresion[
    "dataset_regresion"
]

df_resultados_regresion = resultado_ml_regresion[
    "resultados"
]

modelos_regresion = resultado_ml_regresion[
    "modelos"
]

X_train_reg = resultado_ml_regresion[
    "X_train"
]

X_test_reg = resultado_ml_regresion[
    "X_test"
]

y_train_reg = resultado_ml_regresion[
    "y_train"
]

y_test_reg = resultado_ml_regresion[
    "y_test"
]

display(dataset_regresion_ltv.head())
display(df_resultados_regresion)

print(
    "Shape dataset de regresión:",
    dataset_regresion_ltv.shape
)

print(
    "Carteras train:",
    resultado_ml_regresion["grupos_train"].nunique()
)

print(
    "Carteras test:",
    resultado_ml_regresion["grupos_test"].nunique()
)

,id_cartera,perfil,tipo_financiacion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,hhi_concentracion,ter_cartera,vol_252d_cartera,...,deuda_inicial,ltv_maximo_medio,ltv_maximo_p50,ltv_maximo_p95,ltv_maximo_p99,prob_margin_call,buffer_ltv_medio,severidad_media,rentabilidad_neta_media,drawdown_simulado_medio
0,1,dinamico,linea_credito,0.0,0.0,0.0,1.0,1.00,0.00090,0.135509,...,60000.0,0.682326,0.655201,0.862213,1.042591,0.098,0.117674,0.010137,0.537857,-0.478620
1,1,dinamico,prestamo_amortizable,0.0,0.0,0.0,1.0,1.00,0.00090,0.135509,...,60000.0,0.529869,0.593964,0.652447,0.698322,0.000,0.270131,0.000000,0.591857,-0.478620
2,2,dinamico,linea_credito,0.0,0.0,0.1,0.9,0.82,0.00091,0.125177,...,60000.0,0.674077,0.649862,0.834703,0.989211,0.077,0.125923,0.007134,0.461235,-0.449386
3,2,dinamico,prestamo_amortizable,0.0,0.0,0.1,0.9,0.82,0.00091,0.125177,...,60000.0,0.523977,0.592933,0.644573,0.683613,0.000,0.276023,0.000000,0.515235,-0.449386
4,3,dinamico,linea_credito,0.0,0.0,0.2,0.8,0.68,0.00092,0.115172,...,60000.0,0.666314,0.644662,0.811127,0.943133,0.055,0.133686,0.004825,0.389963,-0.418739


,modelo,mae,rmse,r2
0,XGBoost Regressor,0.001171,0.001936,0.998783
1,Gradient Boosting Regressor,0.001712,0.003229,0.996614
2,Random Forest Regressor,0.004173,0.006079,0.988004


Shape dataset de regresión: (572, 26)
Carteras train: 200
Carteras test: 86


Validación

In [17]:
print("\nValidaciones ML regresión")
print("-" * 40)

print("Filas esperadas:", len(catalogo_carteras) * 2)
print("Filas obtenidas:", len(dataset_regresion_ltv))

sin_fuga = set(
    resultado_ml_regresion["grupos_train"]
).isdisjoint(
    set(resultado_ml_regresion["grupos_test"])
)

print("Train y test sin carteras compartidas:", sin_fuga)

print("\nDistribución del target:")
display(
    dataset_regresion_ltv["ltv_maximo_p95"].describe()
)

print("\nResultados:")
display(df_resultados_regresion)


Validaciones ML regresión
----------------------------------------
Filas esperadas: 572
Filas obtenidas: 572
Train y test sin carteras compartidas: True

Distribución del target:


count    572.000000
mean       0.654207
std        0.055693
min        0.599168
25%        0.605077
50%        0.629931
75%        0.698178
max        0.862213
Name: ltv_maximo_p95, dtype: float64


Resultados:


,modelo,mae,rmse,r2
0,XGBoost Regressor,0.001171,0.001936,0.998783
1,Gradient Boosting Regressor,0.001712,0.003229,0.996614
2,Random Forest Regressor,0.004173,0.006079,0.988004


In [18]:
# ============================================================
# 7. Mejor modelo e importancia de variables
# ============================================================

from sklearn.inspection import permutation_importance


def calcular_importancia_variables_regresion(
    modelo,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42
):
    """
    Calcula la importancia de cada variable mediante permutación.

    La importancia representa cuánto empeora el MAE cuando se altera
    aleatoriamente una variable, manteniendo las demás constantes.

    Una importancia positiva y elevada indica que la variable aporta
    capacidad predictiva al modelo.
    """

    resultado = permutation_importance(
        estimator=modelo,
        X=X_test,
        y=y_test,
        scoring="neg_mean_absolute_error",
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1
    )

    importancia = pd.DataFrame({
        "variable": X_test.columns,
        "importancia_media": resultado.importances_mean,
        "importancia_std": resultado.importances_std
    })

    importancia = (
        importancia
        .sort_values(
            by="importancia_media",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return importancia

In [19]:
# Selección automática del mejor modelo según MAE
mejor_modelo_reg_nombre = (
    df_resultados_regresion
    .iloc[0]["modelo"]
)

mejor_modelo_reg = modelos_regresion[
    mejor_modelo_reg_nombre
]

print(
    "Mejor modelo de regresión según MAE:",
    mejor_modelo_reg_nombre
)

Mejor modelo de regresión según MAE: XGBoost Regressor


Calcular importancia

In [20]:
df_importancia_regresion = (
    calcular_importancia_variables_regresion(
        modelo=mejor_modelo_reg,
        X_test=X_test_reg,
        y_test=y_test_reg,
        n_repeats=30,
        random_state=42
    )
)

print("\nImportancia de variables:")
display(df_importancia_regresion)


Importancia de variables:


,variable,importancia_media,importancia_std
0,tipo_financiacion,0.045471,0.003671
1,vol_252d_cartera,0.015050,0.000994
2,es_63d_95,0.004261,0.000366
3,var_63d_95,0.002354,0.000149
4,rentabilidad_neta_252d,0.002236,0.000147
5,drawdown_63d_cartera,0.001941,0.000159
6,w_AGGU.L,0.000834,0.000083
7,w_XEON.DE,0.000523,0.000079
8,ter_cartera,0.000137,0.000026
9,hhi_concentracion,0.000108,0.000034


Validaciones de importancia

In [21]:

print("\nValidaciones de importancia")
print("-" * 40)

print(
    "Número de variables utilizadas:",
    X_test_reg.shape[1]
)

print(
    "Número de variables evaluadas:",
    len(df_importancia_regresion)
)

print(
    "Variable más importante:",
    df_importancia_regresion.iloc[0]["variable"]
)

print(
    "Importancia máxima:",
    df_importancia_regresion.iloc[0]["importancia_media"]
)

display(df_importancia_regresion.head(10))


Validaciones de importancia
----------------------------------------
Número de variables utilizadas: 16
Número de variables evaluadas: 16
Variable más importante: tipo_financiacion
Importancia máxima: 0.04547075647839877


,variable,importancia_media,importancia_std
0,tipo_financiacion,0.045471,0.003671
1,vol_252d_cartera,0.015050,0.000994
2,es_63d_95,0.004261,0.000366
3,var_63d_95,0.002354,0.000149
4,rentabilidad_neta_252d,0.002236,0.000147
5,drawdown_63d_cartera,0.001941,0.000159
6,w_AGGU.L,0.000834,0.000083
7,w_XEON.DE,0.000523,0.000079
8,ter_cartera,0.000137,0.000026
9,hhi_concentracion,0.000108,0.000034


8. Predicciones del mejor modelo

In [22]:
# ============================================================
# 8. Predicciones y residuos del mejor modelo
# ============================================================

y_pred_reg = mejor_modelo_reg.predict(X_test_reg)

df_predicciones_regresion = pd.DataFrame({
    "ltv_real_p95": y_test_reg.to_numpy(),
    "ltv_predicho_p95": y_pred_reg
})

df_predicciones_regresion["residuo"] = (
    df_predicciones_regresion["ltv_real_p95"]
    - df_predicciones_regresion["ltv_predicho_p95"]
)

df_predicciones_regresion["error_absoluto"] = (
    df_predicciones_regresion["residuo"].abs()
)

df_predicciones_regresion[
    "distancia_predicha_umbral"
] = (
    MARGIN_CALL_THRESHOLD
    - df_predicciones_regresion["ltv_predicho_p95"]
)

df_predicciones_regresion[
    "riesgo_predicho"
] = np.select(
    [
        df_predicciones_regresion[
            "ltv_predicho_p95"
        ] >= 0.80,

        df_predicciones_regresion[
            "ltv_predicho_p95"
        ] >= 0.70,

        df_predicciones_regresion[
            "ltv_predicho_p95"
        ] >= 0.60
    ],
    [
        "critico",
        "alto",
        "moderado"
    ],
    default="bajo"
)

display(df_predicciones_regresion.head())

print("\nDistribución de categorías de riesgo:")
display(
    df_predicciones_regresion[
        "riesgo_predicho"
    ].value_counts()
)

,ltv_real_p95,ltv_predicho_p95,residuo,error_absoluto,distancia_predicha_umbral,riesgo_predicho
0,0.745023,0.746655,-0.001632,0.001632,0.053345,alto
1,0.617828,0.619110,-0.001282,0.001282,0.180890,moderado
2,0.728579,0.727690,0.000889,0.000889,0.072310,alto
3,0.613659,0.613257,0.000402,0.000402,0.186743,moderado
4,0.721359,0.718605,0.002754,0.002754,0.081395,alto



Distribución de categorías de riesgo:


riesgo_predicho
moderado    124
alto         40
bajo          6
critico       2
Name: count, dtype: int64

In [23]:

print("LTV real mínimo:", df_predicciones_regresion["ltv_real_p95"].min())
print("LTV real máximo:", df_predicciones_regresion["ltv_real_p95"].max())

print(
    "LTV predicho mínimo:",
    df_predicciones_regresion["ltv_predicho_p95"].min()
)

print(
    "LTV predicho máximo:",
    df_predicciones_regresion["ltv_predicho_p95"].max()
)

LTV real mínimo: 0.5991812277192219
LTV real máximo: 0.8305223502936588
LTV predicho mínimo: 0.5988349
LTV predicho máximo: 0.82043713


In [24]:
# ============================================================
# 12. Motor Optimización
# ============================================================

def motor_optimizacion(dataset_financiacion):
    resumen = (
        dataset_financiacion
        .groupby([
            "id_cartera",
            "perfil",
            "tipo_financiacion",
            "w_XEON.DE",
            "w_SHY",
            "w_AGGU.L",
            "w_SPY",
            "hhi_concentracion",
            "ter_cartera"
        ])
        .agg(
            retorno_medio_simulado=("retorno_simulado", "mean"),
            retorno_p05=("retorno_simulado", lambda x: x.quantile(0.05)),
            retorno_p95=("retorno_simulado", lambda x: x.quantile(0.95)),
            rentabilidad_neta_media=("rentabilidad_neta_simulada", "mean"),
            prob_margin_call=("margin_call", "mean"),
            ltv_maximo_medio=("ltv_maximo_aproximado", "mean"),
            drawdown_medio=("drawdown_simulado", "mean"),
            drawdown_p05=("drawdown_simulado", lambda x: x.quantile(0.05))
        )
        .reset_index()
    )

    frontera = resumen[
        (resumen["prob_margin_call"] <= 0.05) &
        (resumen["rentabilidad_neta_media"] > 0)
    ].copy()

    frontera = frontera.sort_values(
        ["rentabilidad_neta_media", "prob_margin_call"],
        ascending=[False, True]
    )

    mejor_cartera = frontera.head(10)

    return resumen, frontera, mejor_cartera

In [25]:
resumen_optimizacion, frontera_optima, mejores_carteras = motor_optimizacion(
    dataset_financiacion
)

print("Resumen optimización:", resumen_optimizacion.shape)
print("Frontera óptima:", frontera_optima.shape)

display(mejores_carteras)

Resumen optimización: (572, 17)
Frontera óptima: (434, 17)


,id_cartera,perfil,tipo_financiacion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY,hhi_concentracion,ter_cartera,retorno_medio_simulado,retorno_p05,retorno_p95,rentabilidad_neta_media,prob_margin_call,ltv_maximo_medio,drawdown_medio,drawdown_p05
1,1,dinamico,prestamo_amortizable,0.0,0.0,0.0,1.0,1.00,0.00090,0.646757,-0.107536,1.808427,0.591857,0.0,0.529869,-0.478620,-0.674711
3,2,dinamico,prestamo_amortizable,0.0,0.0,0.1,0.9,0.82,0.00091,0.570145,-0.095244,1.546597,0.515235,0.0,0.523977,-0.449386,-0.643868
23,12,dinamico,prestamo_amortizable,0.0,0.1,0.0,0.9,0.82,0.00096,0.569795,-0.096715,1.560209,0.514835,0.0,0.523285,-0.449187,-0.642785
133,67,dinamico,prestamo_amortizable,0.1,0.0,0.0,0.9,0.82,0.00091,0.561928,-0.095118,1.539393,0.507018,0.0,0.524654,-0.445690,-0.636344
5,3,dinamico,prestamo_amortizable,0.0,0.0,0.2,0.8,0.68,0.00092,0.498883,-0.088657,1.337858,0.443963,0.0,0.518696,-0.418739,-0.612101
25,13,dinamico,prestamo_amortizable,0.0,0.1,0.1,0.8,0.66,0.00097,0.498496,-0.084653,1.337358,0.443526,0.0,0.517870,-0.418444,-0.610743
43,22,dinamico,prestamo_amortizable,0.0,0.2,0.0,0.8,0.68,0.00102,0.498175,-0.088353,1.343824,0.443155,0.0,0.518972,-0.418239,-0.609428
135,68,dinamico,prestamo_amortizable,0.1,0.0,0.1,0.8,0.66,0.00092,0.490985,-0.081277,1.301587,0.436065,0.0,0.519974,-0.414393,-0.602365
153,77,dinamico,prestamo_amortizable,0.1,0.1,0.0,0.8,0.66,0.00097,0.490666,-0.082658,1.316701,0.435696,0.0,0.517633,-0.414170,-0.601381
243,122,dinamico,prestamo_amortizable,0.2,0.0,0.0,0.8,0.68,0.00092,0.483442,-0.082527,1.296132,0.428522,0.0,0.519808,-0.410405,-0.593452


In [26]:
# ============================================================
# 13. Exportación de resultados
# ============================================================

dataset_montecarlo.to_parquet("data/salidas/dataset_montecarlo.parquet", compression="snappy", index=False)
dataset_financiacion.to_parquet("data/salidas/dataset_financiacion.parquet", compression="snappy", index=False)
df_resultados_ml.to_parquet("data/salidas/resultados_modelos_ml.parquet", compression="snappy", index=False)
resumen_optimizacion.to_parquet("data/salidas/resumen_optimizacion.parquet", compression="snappy", index=False)
frontera_optima.to_parquet("data/salidas/frontera_optima.parquet", compression="snappy", index=False)
mejores_carteras.to_parquet("data/salidas/mejores_carteras.parquet", compression="snappy", index=False)

# dataset_montecarlo.to_csv("dataset_montecarlo.csv", index=False)
# dataset_financiacion.to_csv("dataset_financiacion.csv", index=False)
# df_resultados_ml.to_csv("resultados_modelos_ml.csv", index=False)
# resumen_optimizacion.to_csv("resumen_optimizacion.csv", index=False)
# frontera_optima.to_csv("frontera_optima.csv", index=False)
# mejores_carteras.to_csv("mejores_carteras.csv", index=False)

# Resultados del modelo de regresión de LTV

dataset_regresion_ltv.to_parquet(
    "data/salidas/dataset_ml_regresion_ltv.parquet",
    compression="snappy",
    index=False
)

df_resultados_regresion.to_parquet(
    "data/salidas/resultados_modelos_regresion.parquet",
    compression="snappy",
    index=False
)

df_importancia_regresion.to_parquet(
    "data/salidas/importancia_variables_regresion.parquet",
    compression="snappy",
    index=False
)

df_predicciones_regresion.to_parquet(
    "data/salidas/predicciones_modelo_regresion.parquet",
    compression="snappy",
    index=False
)

# Guardar pipeline completo del mejor modelo

import joblib

joblib.dump(
    mejor_modelo_reg,
    "data/salidas/mejor_modelo_regresion_ltv.joblib"
)

print(
    "Mejor modelo guardado:",
    mejor_modelo_reg_nombre
)

print("Exportación final completada.")

Mejor modelo guardado: XGBoost Regressor
Exportación final completada.


## Tablas agregadas para la memoria y para el Notebook 3 de gráficos

Esta sección reproduce, sin cambios de lógica, las agregaciones de la sección de resúmenes del notebook original (antes separadas entre tablas y gráficos en las mismas celdas). Aquí solo se calculan y guardan las tablas; los gráficos se hacen en el Notebook 3.

### 1. Resumen general de datasets

In [27]:
resumen_datasets = pd.DataFrame({
    "dataset": [
        "dataset_mercado",
        "catalogo_carteras",
        "dataset_colateral",
        "dataset_montecarlo",
        "dataset_financiacion",
        "resumen_optimizacion",
        "frontera_optima"
    ],
    "filas": [
        dataset_mercado.shape[0],
        catalogo_carteras.shape[0],
        dataset_colateral.shape[0],
        dataset_montecarlo.shape[0],
        dataset_financiacion.shape[0],
        resumen_optimizacion.shape[0],
        frontera_optima.shape[0]
    ],
    "columnas": [
        dataset_mercado.shape[1],
        catalogo_carteras.shape[1],
        dataset_colateral.shape[1],
        dataset_montecarlo.shape[1],
        dataset_financiacion.shape[1],
        resumen_optimizacion.shape[1],
        frontera_optima.shape[1]
    ]
})

display(resumen_datasets)

,dataset,filas,columnas
0,dataset_mercado,2097,45
1,catalogo_carteras,286,8
2,dataset_colateral,599742,25
3,dataset_montecarlo,286000,21
4,dataset_financiacion,572000,40
5,resumen_optimizacion,572,17
6,frontera_optima,434,17


### 2. Distribución de perfiles de cartera

In [28]:
tabla_perfiles = (
    catalogo_carteras["perfil"]
    .value_counts()
    .rename_axis("perfil")
    .reset_index(name="n_carteras")
)

display(tabla_perfiles)


,perfil,n_carteras
0,conservador,154
1,equilibrado,97
2,dinamico,35


### 3. Evolución histórica del colateral por perfil (serie temporal para gráfico)

In [29]:
colateral_perfil = (
    dataset_colateral
    .groupby(["date", "perfil"])["valor_colateral"]
    .mean()
    .reset_index()
)

display(colateral_perfil.head())


,date,perfil,valor_colateral
0,2017-11-21,conservador,100000.000000
1,2017-11-21,dinamico,100000.000000
2,2017-11-21,equilibrado,100000.000000
3,2017-11-22,conservador,100080.618858
4,2017-11-22,dinamico,99989.786969


### 4. Riesgo histórico por perfil (ventanas 252d / 63d, tal como en el notebook original)

In [30]:
riesgo_historico_perfil = (
    dataset_colateral
    .groupby("perfil")
    .agg(
        valor_colateral_medio=("valor_colateral", "mean"),
        vol_252d_media=("vol_252d_cartera", "mean"),
        drawdown_63d_medio=("drawdown_63d_cartera", "mean"),
        var_63d_95_medio=("var_63d_95", "mean"),
        es_63d_95_medio=("es_63d_95", "mean"),
        rentabilidad_neta_252d_media=("rentabilidad_neta_252d", "mean")
    )
    .reset_index()
)

display(riesgo_historico_perfil)

,perfil,valor_colateral_medio,vol_252d_media,drawdown_63d_medio,var_63d_95_medio,es_63d_95_medio,rentabilidad_neta_252d_media
0,conservador,113754.314369,0.055967,-0.013035,-0.005125,-0.006665,-0.030224
1,dinamico,161391.370120,0.146270,-0.024102,-0.012806,-0.017469,0.057486
2,equilibrado,134024.746418,0.090457,-0.015670,-0.008062,-0.010720,0.010129


### 4B. Riesgo histórico por perfil: volatilidad 21 días y drawdown absoluto

Tabla adicional (no existía en el notebook original) que reproduce exactamente los datos usados en la Ilustración 2 de la memoria ("Riesgo histórico por perfil de cartera"), que en el documento Word usa una ventana de 21 días para la volatilidad y el valor absoluto del drawdown a 63 días, en vez de las ventanas de 252/63 días de la tabla anterior. Se añade aquí para que ese gráfico también quede trazable a una celda de código real.

In [31]:
riesgo_historico_perfil_21d = (
    dataset_colateral
    .groupby("perfil")
    .agg(
        vol_21d_media=("vol_21d_cartera", "mean"),
        drawdown_63d_abs_medio=("drawdown_63d_cartera", lambda x: x.abs().mean())
    )
    .reset_index()
)

display(riesgo_historico_perfil_21d)


,perfil,vol_21d_media,drawdown_63d_abs_medio
0,conservador,0.053530,0.013035
1,dinamico,0.130898,0.024102
2,equilibrado,0.083275,0.015670


### 5. Monte Carlo por perfil (percentiles de retorno simulado)

In [32]:
mc_perfil = (
    dataset_montecarlo
    .groupby("perfil")
    .agg(
        retorno_medio=("retorno_simulado", "mean"),
        retorno_p05=("retorno_simulado", lambda x: x.quantile(0.05)),
        retorno_p50=("retorno_simulado", "median"),
        retorno_p95=("retorno_simulado", lambda x: x.quantile(0.95)),
        drawdown_medio=("drawdown_simulado", "mean"),
        valor_final_medio=("valor_colateral_final", "mean"),
        valor_minimo_medio=("valor_colateral_minimo", "mean")
    )
    .reset_index()
)

display(mc_perfil)

,perfil,retorno_medio,retorno_p05,retorno_p50,retorno_p95,drawdown_medio,valor_final_medio,valor_minimo_medio
0,conservador,0.091846,-0.077359,0.073357,0.313582,-0.152420,109184.626370,95867.342366
1,dinamico,0.427697,-0.074810,0.358384,1.149559,-0.380398,142769.670428,92013.276765
2,equilibrado,0.226811,-0.050310,0.194449,0.612122,-0.253845,122681.084424,94819.381743


### 6. Comparación por tipo de financiación

In [33]:
comparativa_financiacion = (
    dataset_financiacion
    .groupby("tipo_financiacion")
    .agg(
        prob_margin_call=("margin_call", "mean"),
        ltv_maximo_medio=("ltv_maximo_aproximado", "mean"),
        severidad_media=("severidad_margin_call", "mean"),
        buffer_ltv_medio=("buffer_ltv", "mean"),
        rentabilidad_neta_media=("rentabilidad_neta_simulada", "mean"),
        prob_resultado_neto_negativo=("resultado_neto_negativo", "mean")
    )
    .reset_index()
)

display(comparativa_financiacion)


,tipo_financiacion,prob_margin_call,ltv_maximo_medio,severidad_media,buffer_ltv_medio,rentabilidad_neta_media,prob_resultado_neto_negativo
0,linea_credito,0.007559,0.633477,0.00048,0.166523,0.069622,0.469028
1,prestamo_amortizable,0.000000,0.471965,0.00000,0.328035,0.123622,0.311538


### 7. Diagnóstico del modelo de clasificación (datos para ROC, Precision-Recall y matriz de confusión)

El notebook original calculaba estas curvas directamente en la sección de gráficos porque tenía el modelo entrenado y el set de test en memoria. Como el Notebook 3 no vuelve a entrenar nada, aquí se calculan los puntos de las curvas y la matriz de confusión, y se guardan como tablas para que el Notebook 3 solo tenga que graficarlas.

In [34]:
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, confusion_matrix
)

mejor_modelo_nombre = df_resultados_ml.iloc[0]["modelo"]
mejor_modelo = modelos_entrenados[mejor_modelo_nombre]
y_proba = mejor_modelo.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_valor = auc(fpr, tpr)
tabla_roc = pd.DataFrame({"fpr": fpr, "tpr": tpr})
tabla_roc.attrs["modelo"] = mejor_modelo_nombre
tabla_roc.attrs["auc"] = roc_auc_valor

precision, recall, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
tabla_pr = pd.DataFrame({"precision": precision, "recall": recall})

y_pred = (y_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
tabla_confusion = pd.DataFrame(
    cm,
    index=["real_0", "real_1"],
    columns=["pred_0", "pred_1"]
).reset_index().rename(columns={"index": "clase"})

tabla_diagnostico_clasificacion = pd.DataFrame({
    "modelo": [mejor_modelo_nombre],
    "roc_auc": [roc_auc_valor],
    "pr_auc": [ap]
})

display(tabla_diagnostico_clasificacion)
display(tabla_confusion)

tabla_roc.to_parquet("data/salidas/tabla_roc_curve.parquet", compression="snappy", index=False)
tabla_pr.to_parquet("data/salidas/tabla_pr_curve.parquet", compression="snappy", index=False)
tabla_confusion.to_parquet("data/salidas/tabla_confusion_matrix.parquet", compression="snappy", index=False)
tabla_diagnostico_clasificacion.to_parquet("data/salidas/tabla_diagnostico_clasificacion.parquet", compression="snappy", index=False)


,modelo,roc_auc,pr_auc
0,Logistic Regression,0.935336,0.050412


,clase,pred_0,pred_1
0,real_0,141033,29918
1,real_1,66,583


### 8. Distribución de niveles de riesgo predicho

In [35]:
tabla_riesgo_predicho = (
    df_predicciones_regresion[
        "riesgo_predicho"
    ]
    .value_counts()
    .reindex(
        ["bajo", "moderado", "alto", "critico"],
        fill_value=0
    )
    .rename_axis("nivel_riesgo")
    .reset_index(name="n_observaciones")
)

display(tabla_riesgo_predicho)


,nivel_riesgo,n_observaciones
0,bajo,6
1,moderado,124
2,alto,40
3,critico,2


### 9. Composición de las 10 mejores carteras

In [36]:
cols_pesos = ["w_XEON.DE", "w_SHY", "w_AGGU.L", "w_SPY"]

composicion_top = (
    mejores_carteras
    .head(10)
    [["id_cartera", "tipo_financiacion"] + cols_pesos]
)

display(composicion_top)

,id_cartera,tipo_financiacion,w_XEON.DE,w_SHY,w_AGGU.L,w_SPY
1,1,prestamo_amortizable,0.0,0.0,0.0,1.0
3,2,prestamo_amortizable,0.0,0.0,0.1,0.9
23,12,prestamo_amortizable,0.0,0.1,0.0,0.9
133,67,prestamo_amortizable,0.1,0.0,0.0,0.9
5,3,prestamo_amortizable,0.0,0.0,0.2,0.8
25,13,prestamo_amortizable,0.0,0.1,0.1,0.8
43,22,prestamo_amortizable,0.0,0.2,0.0,0.8
135,68,prestamo_amortizable,0.1,0.0,0.1,0.8
153,77,prestamo_amortizable,0.1,0.1,0.0,0.8
243,122,prestamo_amortizable,0.2,0.0,0.0,0.8


### 10. Exportar todas las tablas para la memoria y el Notebook 3

In [37]:
# ============================================================
# 17. Exportar tablas para la memoria
# ============================================================

resumen_datasets.to_parquet(
    "data/salidas/tabla_resumen_datasets.parquet",
    compression="snappy", 
    index=False
)

tabla_perfiles.to_parquet(
    "data/salidas/tabla_perfiles_carteras.parquet",
    compression="snappy",
    index=False
)

riesgo_historico_perfil.to_parquet(
    "data/salidas/tabla_riesgo_historico_perfil.parquet",
    compression="snappy",
    index=False
)

mc_perfil.to_parquet(
    "data/salidas/tabla_montecarlo_perfil.parquet",
    compression="snappy",
    index=False
)

comparativa_financiacion.to_parquet(
    "data/salidas/tabla_comparativa_financiacion.parquet",
    compression="snappy",
    index=False
)

# Benchmark de clasificación
df_resultados_ml.to_parquet(
    "data/salidas/tabla_resultados_ml_clasificacion.parquet",
    compression="snappy",
    index=False
)

# Modelo principal de regresión
df_resultados_regresion.to_parquet(
    "data/salidas/tabla_resultados_ml_regresion.parquet",
    compression="snappy",
    index=False
)

df_importancia_regresion.to_parquet(
    "data/salidas/tabla_importancia_variables_regresion.parquet",
    compression="snappy",
    index=False
)

df_predicciones_regresion.to_parquet(
    "data/salidas/tabla_predicciones_regresion.parquet",
    compression="snappy",
    index=False
)

tabla_riesgo_predicho.to_parquet(
    "data/salidas/tabla_riesgo_predicho.parquet",
    compression="snappy",
    index=False
)

# Optimización
mejores_carteras.to_parquet(
    "data/salidas/tabla_mejores_carteras.parquet",
    compression="snappy",
    index=False
)

frontera_optima.to_parquet(
    "data/salidas/tabla_frontera_optima.parquet",
    compression="snappy",
    index=False
)
colateral_perfil.to_parquet(
    "data/salidas/tabla_colateral_perfil_evolucion.parquet", 
    compression="snappy",  
    index=False
)
riesgo_historico_perfil_21d.to_parquet(
    "data/salidas/tabla_riesgo_historico_perfil_21d.parquet", 
    compression="snappy", 
    index=False
)
print(
    "Tablas de mercado, financiación, clasificación, "
    "regresión y optimización exportadas correctamente."
)

Tablas de mercado, financiación, clasificación, regresión y optimización exportadas correctamente.


### 11. Validación final del pipeline

In [38]:
# ============================================================
# 19. Validación final del pipeline
# ============================================================

validaciones_finales = {
    "dataset_regresion_572_filas": (
        len(dataset_regresion_ltv) == 572
    ),
    "sin_fuga_entre_train_test": sin_fuga,
    "resultados_regresion_generados": (
        not df_resultados_regresion.empty
    ),
    "importancias_generadas": (
        len(df_importancia_regresion)
        == X_test_reg.shape[1]
    ),
    "predicciones_generadas": (
        len(df_predicciones_regresion)
        == len(y_test_reg)
    ),
    "modelo_entrenado": (
        mejor_modelo_reg is not None
    ),
    "frontera_optima_generada": (
        not frontera_optima.empty
    )
}

df_validaciones_finales = (
    pd.Series(
        validaciones_finales,
        name="resultado"
    )
    .rename_axis("validacion")
    .reset_index()
)

display(df_validaciones_finales)

if not df_validaciones_finales["resultado"].all():
    raise ValueError(
        "Hay una o más validaciones finales que no se cumplen."
    )

print(
    "\nPipeline técnico validado correctamente. "
    "La fase técnica puede considerarse cerrada."
)

,validacion,resultado
0,dataset_regresion_572_filas,True
1,sin_fuga_entre_train_test,True
2,resultados_regresion_generados,True
3,importancias_generadas,True
4,predicciones_generadas,True
5,modelo_entrenado,True
6,frontera_optima_generada,True



Pipeline técnico validado correctamente. La fase técnica puede considerarse cerrada.


In [39]:
df_validaciones_finales.to_parquet(
    "data/salidas/tabla_validaciones_finales.parquet",
    compression="snappy",
    index=False
)